## Dipole far-field magnetic geometry, simple tests

Checks:
* Do magnetic dipole field lines look reasonable?
* Bounce-averaged particle drift approximately scale as $v^2$ ?
* Ratio of exact and approximate drift frequencies agrees with Kesner & Hastie (2002) Figure 1?

This notebook also shows
* what bounce-integral grid resolution is needed to get good agreement with exact result
* number of sample points to use for dipole field line tracing

References:
* Mishchenko, Plunk, Helander (2018, J. Plasma Phys.)
* Kesner & Hastie (2002, Phys. Plasmas)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from datetime import datetime

import dredge as dr

%load_ext autoreload
%autoreload 2

# Visualize dipole field geometry

In [ ]:
for ii, req in enumerate(np.linspace(1,50,20)):
    field = dr.field.DipoleFieldLine(
        I=1, r0=1, req=req,
        ds=0.2, n_steps=int(200*req/30),
        axis_r = 1, axis_z = 2,
    )
    plt.plot(field.r, field.z, c=f'C{ii}', lw=0.5)
    plt.plot(field.r, -field.z, c=f'C{ii}', lw=0.5)
    plt.plot(-field.r, field.z, c=f'C{ii}', lw=0.5)
    plt.plot(-field.r, -field.z, c=f'C{ii}', lw=0.5)
plt.gca().set_aspect('equal')
plt.xlabel('y [cm]')
plt.ylabel('z [cm]')
plt.show()

In [ ]:
req = 50  # cm, equatorial plane radius at which to trace lines

field = dr.field.DipoleFieldLine(
        I=1e17, r0=1, req=req,
        ds=0.2, n_steps=int(200*req/30),
        axis_r = 1, axis_z = 2,
)

plt.figure(figsize=(2,1.5))
plt.plot(field.s, field.By, label='By')
plt.plot(field.s, field.Bz, label='Bz')
plt.plot(field.s, -field.By, label='-By', c='C0', ls=':')
plt.plot(field.s, -field.Bz, label='-Bz', c='C1', ls=':')
plt.plot(field.s, field.Bmag, label='|B|', c='k')
plt.yscale('log')
plt.ylim(ymin=1e-1)
plt.legend()
plt.xlabel('Arc length s [cm]')
plt.ylabel('Magnetic field [Gauss]')
plt.show()

In [ ]:
plt.figure(figsize=(2,1.5))
plt.plot(field.s, field.r/field.r[0], label=r'$r/r_0$')
plt.plot(field.s, 1./(field.Bmag/field.Bmag[0])**0.5, label=r'$1/\sqrt{B/B_0}$')
plt.legend()
plt.xlabel('Arc length s [cm]')
plt.ylabel('Ratio for mode number scaling')
plt.show()

In [ ]:
plt.figure(figsize=(2,1.5))
plt.plot(field.s, np.sqrt(field.dbhat_ds_y**2 + field.dbhat_ds_z**2))
plt.xlabel('Arc length $s$ [cm]')
plt.ylabel(r'Curvature magnitude $\kappa$ [1/cm]')
plt.show()

## Setup dipole field and species for bounce averaging

In [ ]:
from dredge.const import CLIGHT, M_ELECTRON, M_PROTON, ERG_PER_EV, Q_ELEMENTARY, GAUSS_PER_TESLA

In [ ]:
Ti = 100 * ERG_PER_EV  # arbitrary
vthi = np.sqrt(2*Ti/M_PROTON)

# using odd number of points to vprll_vec to get vprll=0 exactly,
# stress test the mu=0 edge case
vperp_vec = np.linspace(0, 4*vthi, 100)  # cm/s
vprll_vec = np.linspace(-4*vthi, 4*vthi, 201)  # cm/s
vperp, vprll = np.meshgrid(vperp_vec, vprll_vec, indexing='ij')
df = dr.vdf.bimaxwellian(vperp, vprll, vthi, vthi)  # isotropic maxwellian, (cm/s)^-3

In [ ]:
ion = dr.species.KineticVDFGrid(
    mass=M_PROTON,
    charge=Q_ELEMENTARY,
    vperp_vec=vperp_vec,
    vprll_vec=vprll_vec,
    df=df,
)

In [ ]:
field = dr.field.DipoleFieldLine(
        I=1e17, r0=1, req=req,
        ds=0.2, n_steps=int(200*req/30),
        axis_r = 1, axis_z = 2,
)

In [ ]:
# just a dummy argument
# not used for bounce-average calculation tests
solve_grid = dr.chi.WaveGrid(
    k_vec_global = np.linspace(1e-6, 1, 10) / ion.rLs(field.Bmag[0]),
    omega_re_vec_global = np.linspace(1e-6, 1, 11) * ion.Omcs(field.Bmag[0]),
    omega_im_vec_global = np.linspace(1e-6, 1, 12) * ion.Omcs(field.Bmag[0]),
    proc_layout = (1,1,1),
)

In [ ]:
calc = dr.chi.BounceAvgESPerp(
    grid = solve_grid,
    species = ion,
    field = field,
)

## Compute bounce-averaged drift

Compare exact calculation of $\bar{\omega}_{di}$
to an approximation given by page 11 of Mishchenko+ (2018).

In [ ]:
started = datetime.now()

calc.setup_bounce_average(NS_RESOLUTION = 300)
# additional setup for bounce average
# that is not fully incorporated into my code
rsamp = field.query_r_at(calc.ssamp)  # shape (vperp,vprll,s)

print('done setting up for bounce avg, elapsed', datetime.now()-started)

In [ ]:
# compute bounce-averaged (omega_drift / m) where m = azimuthal mode number
# this includes k_\perp(s) structure into the bounce average
calc._reset_timers()
omega_m_gradB_BA = calc.bounce_average_norm_raw(calc.v_gradB[0]/rsamp, norm=True)
omega_m_curv_BA  = calc.bounce_average_norm_raw(calc.v_curv[0]/rsamp,  norm=True)
omega_m_drift_BA = omega_m_gradB_BA + omega_m_curv_BA

In [ ]:
# flux label of selected field line
psi = field.M / field.req

# need -1 to reconcile my (x,y,z) coordinates with Mishchenko's (r,z,phi) coordinates
# and need factor of CLIGHT to convert SI into CGS
# Mishchenko+ (2018 JPP), Section 5, top of page 11
omega_m_drift_BA_expected = -1 * (8/3) * calc.E * CLIGHT / (ion.charge * psi)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(3.375,3), sharex=True, sharey=True)

plt.sca(axes[0])
plt.title(r'$\bar{\omega}_{di} / k_\varphi = \bar{\omega}_{di} / m$ [rad/µs]')
plt.imshow(
    omega_m_drift_BA * 1e-6,
    extent=(ion.vprll_vec[0]/ion.vth_prll, ion.vprll_vec[-1]/ion.vth_prll,
            ion.vperp_vec[0]/ion.vth_perp, ion.vperp_vec[-1]/ion.vth_perp,),
    origin='lower',
    cmap='turbo', vmin=0, vmax=-3,
)

plt.sca(axes[1])
plt.title(r'Mishchenko approx: $4 m v^2 / (3 q_i \psi)$ [rad/µs]', fontsize='small')
plt.imshow(
    omega_m_drift_BA_expected * 1e-6,
    extent=(ion.vprll_vec[0]/ion.vth_prll, ion.vprll_vec[-1]/ion.vth_prll,
            ion.vperp_vec[0]/ion.vth_perp, ion.vperp_vec[-1]/ion.vth_perp,),
    origin='lower',
    cmap='turbo', vmin=0, vmax=-3,
)

plt.sca(axes[2])
plt.title(r'(exact - approx) / approx')
plt.imshow(
    omega_m_drift_BA_expected / omega_m_drift_BA - 1,
    extent=(ion.vprll_vec[0]/ion.vth_prll, ion.vprll_vec[-1]/ion.vth_prll,
            ion.vperp_vec[0]/ion.vth_perp, ion.vperp_vec[-1]/ion.vth_perp,),
    origin='lower',
    cmap='RdBu', vmin=-0.25, vmax=0.25,
)

for ax in axes:
    plt.sca(ax)
    plt.colorbar()
    plt.ylabel(r'$v_\perp / v_\mathrm{th}$')

axes[-1].set_xlabel(r'$v_\parallel / v_\mathrm{th}$')
plt.subplots_adjust(hspace=0.4)
plt.show()

## Replicate Kesner & Hastie (2002) Figure 1

Mishchenko cites Kesner & Hastie (2002) for the bounce-averaged drift approximation.

Kesner & Hastie (2002, Figure 1) shows the ratio of exact to approximate drifts.
Appendix of the same paper discusses effect of the approximation on stability boundary.

Data from Kesner's figure was extracted using WebPlotDigitizer at https://automeris.io/v4/

In [ ]:
kesner_x, kesner_y = np.loadtxt('../data/kesner2002_figure1.csv', delimiter=',').T

In [ ]:
# select a cut in velocity space near vprll=0 to get maximum range of mu/E
jsel = 110
print("select vprll/vth =", ion.vprll_vec[jsel]/ion.vth_prll)

In [ ]:
plt.plot(
    (calc.mu/calc.E)[:,jsel] * calc.B0,
    omega_m_drift_BA[:,jsel] / (omega_m_drift_BA_expected[:,jsel] * 3/2),
    '.-', markersize=2,
    label='dredge code'
)

plt.plot(
    kesner_x, kesner_y,
    '-', alpha=0.5, lw=2, zorder=99, c='k',
    label='Kesner/Hastie (2002) Figure 1'
)

plt.axhline(2/3, c='k', ls='--')
plt.xlim(0, 1)
plt.ylim(0, 0.8)
plt.grid()
plt.legend()
plt.xlabel(r'Scaled pitch angle $\lambda B_0 = \mu B_0/E$ at equator')
plt.ylabel(r"$\bar{\omega}_{di} / (\epsilon \hat{\omega}_{di}/T_i)"
           + r"\;=\; \bar{\omega}_{di} / (2mv^2/q_i \psi)$")
plt.show()

In [ ]:
for ns_resolution in [30, 100, 300, 1000]:

    print('ns_resolution =', ns_resolution)
    
    started = datetime.now()
    calc.setup_bounce_average(NS_RESOLUTION = ns_resolution)
    print('... done setting up for bounce avg, elapsed', datetime.now()-started)

    calc._reset_timers()
    # compute bounce-averaged (omega_drift / m) where m = azimuthal mode number
    # this includes k_\perp(s) structure into the bounce average
    omega_m_gradB_BA = calc.bounce_average_norm_raw(calc.v_gradB[0]/calc.rsamp, norm=True)
    omega_m_curv_BA  = calc.bounce_average_norm_raw(calc.v_curv[0]/calc.rsamp,  norm=True)
    omega_m_drift_BA = omega_m_gradB_BA + omega_m_curv_BA
    print('... done bounce avg, elapsed', datetime.now()-started)
    
    plt.plot(
        (calc.mu/calc.E)[:,jsel] * calc.B0,
        omega_m_drift_BA[:,jsel] / (omega_m_drift_BA_expected[:,jsel] * 3/2),
        '.-', markersize=2, label=r'$N_s =$' + f'{ns_resolution:d}',
    )

plt.plot(
    kesner_x, kesner_y,
    '-', alpha=0.5, lw=2, zorder=99, c='k',
    label='Kesner/Hastie (2002) Figure 1'
)

plt.axhline(2/3, c='k', ls='--')
plt.xlim(0, 1)
plt.ylim(0, 0.8)
plt.grid()
plt.legend()
plt.xlabel(r'Scaled pitch angle $\lambda B_0 = \mu B_0/E$ at equator')
plt.ylabel(r"$\bar{\omega}_{di} / (\epsilon \hat{\omega}_{di}/T_i)"
           + r"\;=\; \bar{\omega}_{di} / (2mv^2/q_i \psi)$")
plt.show()